# Batch Query Test Against Known-Pose Map

This notebook localizes test/query images against the known-pose reconstruction produced by `reconstruction_known_pose.ipynb`. Query images can come from any camera as long as their basenames exist in `metadata/poses.json`.

The map and query cameras do not need to share the same camera_name. Ground truth is matched by image basename.

## 1. Configuration

In [2]:
from pathlib import Path
from datetime import datetime
import copy
import json
import random
import shutil

import numpy as np
import pandas as pd
import pycolmap
import torch

from hloc import extract_features, match_features, pairs_from_retrieval
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster
from hloc.utils.parsers import parse_retrieval

from simulation_pose_utils import (
    image_basename,
    load_json,
    load_pose_records,
    metadata_pose_center,
    pycolmap_transform_rt,
    quat_wxyz_to_rotmat,
)

# Dataset with query metadata.
dataset_root = Path('../datasets/sim_test_images_19_may')
intrinsics_json = dataset_root / 'metadata/intrinsics_pinhole.json'

# Query image folder. Can contain any camera as long as basenames exist in poses.json.
query_source_dir = dataset_root / 'test'
query_glob = '*.png'
max_queries = 50
random_seed = 42

# Known-pose map bundle created by reconstruction_known_pose.ipynb.
map_bundle_root = Path('../outputs/sim-19-may-bundle')
sfm_model_root = map_bundle_root / 'sfm'
db_features = map_bundle_root / 'features.h5'
db_global_features = map_bundle_root / 'global-feats-netvlad.h5'

# Query outputs.
results_dir = map_bundle_root / 'query_batch_results_v2_19_may'
query_cache_dir = map_bundle_root / 'query_batch_v2_19_may_cache'
results_dir.mkdir(parents=True, exist_ok=True)
query_cache_dir.mkdir(parents=True, exist_ok=True)

# Localization parameters.
num_loc = 10
max_error = 12
overwrite_query_features = False

feature_conf = copy.deepcopy(extract_features.confs['superpoint_max'])
retrieval_conf = extract_features.confs['netvlad']
matcher_conf = match_features.confs['superpoint+lightglue']

# Windows/sandbox-safe HLoc execution: avoid multiprocessing DataLoader workers.
# Keep class-compatible with Kornia's DataLoader[Any] annotation.
_original_dataloader = torch.utils.data.DataLoader
class _SingleProcessDataLoader(_original_dataloader):
    @classmethod
    def __class_getitem__(cls, item):
        return cls

    def __init__(self, *args, **kwargs):
        kwargs['num_workers'] = 0
        kwargs['pin_memory'] = False
        super().__init__(*args, **kwargs)
torch.utils.data.DataLoader = _SingleProcessDataLoader

for path in [query_source_dir, sfm_model_root, db_features, db_global_features]:
    if not path.exists():
        raise FileNotFoundError(path)

intrinsics_cfg = load_json(intrinsics_json)['cameras'][0]
print(f'Dataset: {dataset_root}')
print(f'Query dir: {query_source_dir}')
print(f'Map bundle: {map_bundle_root}')
print(f'SfM model: {sfm_model_root}')
print(f'Intrinsics: {intrinsics_cfg["model"]} {intrinsics_cfg["width"]}x{intrinsics_cfg["height"]} params={intrinsics_cfg["params"]}')

Dataset: ..\datasets\sim_test_images_19_may
Query dir: ..\datasets\sim_test_images_19_may\test
Map bundle: ..\outputs\sim-19-may-bundle
SfM model: ..\outputs\sim-19-may-bundle\sfm
Intrinsics: PINHOLE 1920x1080 params=[879.6779270567265, 879.6779270567265, 960.0, 540.0]


## 2. Load Map and Query Ground Truth

In [3]:
model = pycolmap.Reconstruction(sfm_model_root)
print(model.summary())

all_pose_records, pose_by_name, _ = load_pose_records(dataset_root, camera_names=None)
query_paths_all = sorted(p for p in query_source_dir.glob(query_glob) if p.is_file())
if max_queries is not None and max_queries < len(query_paths_all):
    rng = random.Random(random_seed)
    query_paths = sorted(rng.sample(query_paths_all, max_queries))
else:
    query_paths = query_paths_all

missing_metadata = [p.name for p in query_paths if p.name not in pose_by_name]
if missing_metadata:
    raise ValueError(f'{len(missing_metadata)} query images have no poses.json metadata. Examples: {missing_metadata[:10]}')

query_camera_distribution = {}
for p in query_paths:
    cam = pose_by_name[p.name]['camera_name']
    query_camera_distribution[cam] = query_camera_distribution.get(cam, 0) + 1

references = sorted(image.name for image in model.images.values())
print(f'All query images found: {len(query_paths_all)}')
print(f'Queries selected: {len(query_paths)}')
print(f'Random seed: {random_seed if max_queries is not None else None}')
print(f'Query camera distribution: {query_camera_distribution}')
print(f'Reference images in map: {len(references)}')

Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 185
	num_reg_frames = 185
	num_images = 185
	num_points3D = 52096
	num_observations = 153252
	mean_track_length = 2.94172
	mean_observations_per_image = 828.389
	mean_reprojection_error = 1.3396
All query images found: 37
Queries selected: 37
Random seed: 42
Query camera distribution: {'front': 37}
Reference images in map: 185


## 3. Helper Functions

In [4]:
def extract_on_cpu(conf, image_root, image_list, feature_path, overwrite=True):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    original = torch.cuda.is_available
    torch.cuda.is_available = lambda: False
    try:
        return extract_features.main(
            conf,
            image_root,
            image_list=image_list,
            feature_path=feature_path,
            overwrite=overwrite,
        )
    finally:
        torch.cuda.is_available = original


def resolve_reference_ids(model, names):
    ref_ids = []
    missing = []
    for name in names:
        image = model.find_image_with_name(name)
        if image is None:
            basename = image_basename(name)
            for candidate in model.images.values():
                if image_basename(candidate.name) == basename:
                    image = candidate
                    break
        if image is None:
            missing.append(name)
        else:
            ref_ids.append(image.image_id)
    if missing:
        raise ValueError('Retrieved images not registered in model: ' + ', '.join(missing))
    return ref_ids


CARLA_TO_COLMAP_S = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.0, -1.0],
    [1.0, 0.0, 0.0],
], dtype=np.float64)
COLMAP_TO_CARLA_S = np.linalg.inv(CARLA_TO_COLMAP_S)


def wrap_angle_deg(angle):
    return ((float(angle) + 180.0) % 360.0) - 180.0


def heading_error_deg(est_heading, gt_heading):
    return abs(wrap_angle_deg(float(est_heading) - float(gt_heading)))


def colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh):
    # PnP/pycolmap returns COLMAP right-handed world-to-camera rotation.
    # Convert it back to CARLA left-handed camera-to-world rotation before extracting yaw.
    r_cw_rh = np.asarray(r_wc_rh, dtype=np.float64).T
    r_cw_lh = COLMAP_TO_CARLA_S @ r_cw_rh @ CARLA_TO_COLMAP_S
    return wrap_angle_deg(np.degrees(np.arctan2(r_cw_lh[1, 0], r_cw_lh[0, 0])))


def metadata_record_to_carla_yaw_deg(record):
    r_wc_rh = quat_wxyz_to_rotmat([record['qw'], record['qx'], record['qy'], record['qz']])
    return colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh)


metadata_yaw_errors = []
for query_path in query_paths:
    rec = pose_by_name[query_path.name]
    metadata_yaw_errors.append(heading_error_deg(metadata_record_to_carla_yaw_deg(rec), rec['yaw_deg']))
metadata_yaw_errors = np.array(metadata_yaw_errors, dtype=np.float64)
print('Metadata quaternion -> CARLA yaw sanity check')
print(f'  count: {len(metadata_yaw_errors)}')
print(f'  mean error: {metadata_yaw_errors.mean():.9f} deg')
print(f'  max error: {metadata_yaw_errors.max():.9f} deg')
if metadata_yaw_errors.max() >= 1e-3:
    raise AssertionError('Metadata quaternion to CARLA yaw conversion sanity check failed.')


localizer_conf = {
    'estimation': {'ransac': {'max_error': max_error}},
    'refinement': {'refine_focal_length': False, 'refine_extra_params': False},
}
localizer = QueryLocalizer(model, localizer_conf)
camera = pycolmap.Camera(
    model=intrinsics_cfg['model'],
    width=int(intrinsics_cfg['width']),
    height=int(intrinsics_cfg['height']),
    params=np.array(intrinsics_cfg['params'], dtype=float),
)

Metadata quaternion -> CARLA yaw sanity check
  count: 37
  mean error: 0.000001875 deg
  max error: 0.000005659 deg


## 4. Batch Query Localization

In [5]:
localization_results = []

for idx, query_path in enumerate(query_paths, start=1):
    query_basename = query_path.name
    query_gt = pose_by_name[query_basename]
    query_gt_center = metadata_pose_center(query_gt)
    query_dst = query_cache_dir / query_basename
    shutil.copy2(query_path, query_dst)
    query_rel = f'{query_cache_dir.name}/{query_basename}'

    print(f'\n[{idx}/{len(query_paths)}] {query_basename} camera={query_gt["camera_name"]}')

    stem = Path(query_basename).stem
    query_features = results_dir / f'{stem}-features.h5'
    query_global_features = results_dir / f'{stem}-global-feats-netvlad.h5'
    query_matches = results_dir / f'{stem}-matches.h5'
    loc_pairs = results_dir / f'{stem}-pairs-query-netvlad.txt'

    try:
        extract_on_cpu(retrieval_conf, map_bundle_root, [query_rel], query_global_features, overwrite=overwrite_query_features)
        extract_on_cpu(feature_conf, map_bundle_root, [query_rel], query_features, overwrite=overwrite_query_features)

        pairs_from_retrieval.main(
            descriptors=query_global_features,
            output=loc_pairs,
            num_matched=min(num_loc, len(references)),
            query_list=[query_rel],
            db_list=references,
            db_descriptors=db_global_features,
        )

        match_features.main(
            matcher_conf,
            loc_pairs,
            features=query_features,
            features_ref=db_features,
            matches=query_matches,
            overwrite=True,
        )

        retrieval_dict = parse_retrieval(loc_pairs)
        retrieved_names = retrieval_dict[query_rel]
        ref_ids = resolve_reference_ids(model, retrieved_names)

        ret, log = pose_from_cluster(localizer, query_rel, camera, ref_ids, query_features, query_matches)
        if ret is None:
            print('  pose estimation failed')
            localization_results.append({
                'image_name': query_basename,
                'camera_name': query_gt['camera_name'],
                'success': False,
                'error': 'pose_from_cluster returned None',
                'retrieved': retrieved_names,
            })
            continue

        R_wc_rh, tvec = pycolmap_transform_rt(ret['cam_from_world'])
        estimated_center = -R_wc_rh.T @ tvec
        position_error_m = float(np.linalg.norm(estimated_center - query_gt_center))

        estimated_heading = colmap_world_to_camera_to_carla_yaw_deg(R_wc_rh)
        gt_heading = float(query_gt['yaw_deg'])
        heading_error = heading_error_deg(estimated_heading, gt_heading)

        result = {
            'image_name': query_basename,
            'camera_name': query_gt['camera_name'],
            'capture_id': int(query_gt['capture_id']),
            'success': True,
            'num_inliers': int(ret['num_inliers']),
            'position_error_m': position_error_m,
            'estimated_center_x': float(estimated_center[0]),
            'estimated_center_y': float(estimated_center[1]),
            'estimated_center_z': float(estimated_center[2]),
            'gt_center_x': float(query_gt_center[0]),
            'gt_center_y': float(query_gt_center[1]),
            'gt_center_z': float(query_gt_center[2]),
            'estimated_heading_deg': estimated_heading,
            'ground_truth_yaw_deg': gt_heading,
            'heading_error_deg': float(heading_error),
            'retrieved': retrieved_names,
        }
        localization_results.append(result)
        print(f"  success inliers={result['num_inliers']} pos_err={position_error_m:.3f} m heading_err={heading_error:.2f} deg")

    except Exception as exc:
        print(f'  error: {exc}')
        localization_results.append({
            'image_name': query_basename,
            'camera_name': query_gt['camera_name'],
            'capture_id': int(query_gt['capture_id']),
            'success': False,
            'error': str(exc),
        })

print('\nBatch localization finished')

[2026/05/19 18:32:06 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}



[1/37] 000001_front_f00000590.png camera=front


100%|██████████| 1/1 [00:01<00:00,  1.28s/it]
[2026/05/19 18:32:14 hloc INFO] Finished exporting features.
[2026/05/19 18:32:14 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


Loaded SuperPoint model


c:\users\ilker\desktop\bitirme-project\hierarchical-localization\hloc\extractors\..\..\third_party\SuperGluePretrainedNetwork\models\superpoint.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

...

  success inliers=925 pos_err=0.811 m heading_err=3.73 deg

[37/37] 000037_front_f00001803.png camera=front


100%|██████████| 1/1 [00:01<00:00,  1.10s/it]
[2026/05/19 18:37:41 hloc INFO] Finished exporting features.
[2026/05/19 18:37:41 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


Loaded SuperPoint model


100%|██████████| 1/1 [00:02<00:00,  2.29s/it]
[2026/05/19 18:37:43 hloc INFO] Finished exporting features.
[2026/05/19 18:37:43 hloc INFO] Extracting image pairs from a retrieval database.
[2026/05/19 18:37:43 hloc INFO] Found 10 pairs.
[2026/05/19 18:37:43 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 10/10 [00:00<00:00, 17.38it/s]
[2026/05/19 18:37:44 hloc INFO] Finished exporting matches.


  success inliers=829 pos_err=0.567 m heading_err=5.19 deg

Batch localization finished


## 5. Results Summary

In [6]:
results_df = pd.DataFrame(localization_results)
display(results_df)

successful = results_df[results_df['success'] == True]
failed = results_df[results_df['success'] != True]

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'dataset_root': str(dataset_root),
    'query_source_dir': str(query_source_dir),
    'map_bundle_root': str(map_bundle_root),
    'sfm_model_root': str(sfm_model_root),
    'total_queries': int(len(results_df)),
    'successful_queries': int(len(successful)),
    'failed_queries': int(len(failed)),
    'success_rate': float(len(successful) / max(1, len(results_df))),
    'num_retrieved': num_loc,
    'ransac_max_error_px': max_error,
}

if len(successful) > 0:
    summary['position_error_m'] = {
        'mean': float(successful['position_error_m'].mean()),
        'median': float(successful['position_error_m'].median()),
        'max': float(successful['position_error_m'].max()),
    }
    summary['inliers'] = {
        'mean': float(successful['num_inliers'].mean()),
        'median': float(successful['num_inliers'].median()),
        'min': int(successful['num_inliers'].min()),
    }
    if 'heading_error_deg' in successful:
        summary['heading_error_deg'] = {
            'mean': float(successful['heading_error_deg'].mean()),
            'median': float(successful['heading_error_deg'].median()),
            'max': float(successful['heading_error_deg'].max()),
        }

print(json.dumps(summary, indent=2))

details_csv = results_dir / 'query_batch_details.csv'
summary_json = results_dir / 'query_batch_summary.json'
results_df.to_csv(details_csv, index=False)
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Details CSV: {details_csv}')
print(f'Summary JSON: {summary_json}')

,image_name,camera_name,capture_id,success,num_inliers,position_error_m,estimated_center_x,estimated_center_y,estimated_center_z,gt_center_x,gt_center_y,gt_center_z,estimated_heading_deg,ground_truth_yaw_deg,heading_error_deg,retrieved
0,000001_front_f00000590.png,front,1,True,1555,0.014255,-7.780672,-2.205363,10.614608,-7.789151,-2.199965,10.624717,98.244906,98.272072,0.027166,"[images/000024_front_f00038565.png, images/000..."
1,000002_front_f00000634.png,front,2,True,1265,0.243391,-2.486902,-2.215457,8.185712,-2.250390,-2.202351,8.129769,109.848377,109.900536,0.052158,"[images/000025_front_f00038653.png, images/000..."
2,000003_front_f00000672.png,front,3,True,1903,0.482880,11.261900,-2.208394,2.688573,11.699672,-2.202441,2.484875,112.838345,112.781738,0.056607,"[images/000031_front_f00039026.png, images/000..."
3,000004_front_f00000690.png,front,4,True,1331,0.516385,20.433455,-2.203110,-0.412033,20.945584,-2.202205,-0.478196,103.601691,102.221634,1.380057,"[images/000035_front_f00039220.png, images/000..."
4,000005_front_f00000725.png,front,5,True,1047,0.649964,40.577299,-2.201244,-7.969250,41.134189,-2.202037,-8.304401,115.407368,117.056816,1.649448,"[images/000041_front_f00039604.png, images/000..."
5,000006_front_f00000753.png,front,6,True,1070,0.609397,58.523142,-2.200751,-15.020332,59.111687,-2.203734,-15.178349,105.448160,105.867096,0.418936,"[images/000048_front_f00039934.png, images/000..."
6,000007_front_f00000771.png,front,7,True,652,0.664948,71.356540,-2.198737,-18.758311,71.997605,-2.202128,-18.934890,106.295221,106.299065,0.003844,"[images/000052_front_f00040126.png, images/000..."
7,000008_front_f00000794.png,front,8,True,590,0.676932,88.248817,-2.194695,-25.205282,88.859001,-2.202073,-25.498298,115.605177,115.663055,0.057878,"[images/000053_front_f00040162.png, images/000..."
8,000009_front_f00000823.png,front,9,True,335,0.706870,110.373105,-2.181914,-34.245752,111.024932,-2.202032,-34.518483,112.244863,112.284019,0.039157,"[images/000066_front_f00041040.png, images/000..."
9,000010_front_f00000848.png,front,10,True,690,0.750979,130.256519,-2.196721,-42.186464,130.958601,-2.201257,-42.452977,110.952254,110.929039,0.023215,"[images/000062_front_f00040751.png, images/000..."


{
  "timestamp": "2026-05-19T18:38:16",
  "dataset_root": "..\\datasets\\sim_test_images_19_may",
  "query_source_dir": "..\\datasets\\sim_test_images_19_may\\test",
  "map_bundle_root": "..\\outputs\\sim-19-may-bundle",
  "sfm_model_root": "..\\outputs\\sim-19-may-bundle\\sfm",
  "total_queries": 37,
  "successful_queries": 37,
  "failed_queries": 0,
  "success_rate": 1.0,
  "num_retrieved": 10,
  "ransac_max_error_px": 12,
  "position_error_m": {
    "mean": 0.4976847720604439,
    "median": 0.5366444500858384,
    "max": 0.8110124035723704
  },
  "inliers": {
    "mean": 724.6216216216217,
    "median": 652.0,
    "min": 56
  },
  "heading_error_deg": {
    "mean": 0.7857022925085199,
    "median": 0.05660703500598174,
    "max": 5.191871464784185
  }
}
Details CSV: ..\outputs\sim-19-may-bundle\query_batch_results_v2_19_may\query_batch_details.csv
Summary JSON: ..\outputs\sim-19-may-bundle\query_batch_results_v2_19_may\query_batch_summary.json


## 6. VisualLocalization.net Threshold Format

In [7]:
benchmark_thresholds = [
    (0.25, 2.0),
    (0.50, 5.0),
    (5.00, 10.0),
]

total_queries = len(results_df)
successful_eval = successful.dropna(subset=['position_error_m', 'heading_error_deg']).copy()

benchmark_parts = []
benchmark_rows = []
for pos_thr, rot_thr in benchmark_thresholds:
    passed = successful_eval[
        (successful_eval['position_error_m'] <= pos_thr)
        & (successful_eval['heading_error_deg'] <= rot_thr)
    ]
    count = int(len(passed))
    percent_total = 100.0 * count / max(1, total_queries)
    percent_successful = 100.0 * count / max(1, len(successful_eval))
    benchmark_parts.append(f'{percent_total:.1f}')
    benchmark_rows.append({
        'position_threshold_m': pos_thr,
        'heading_threshold_deg': rot_thr,
        'passed': count,
        'total_queries': int(total_queries),
        'successful_evaluated_queries': int(len(successful_eval)),
        'percent_of_all_queries': percent_total,
        'percent_of_successful_queries': percent_successful,
    })

benchmark_df = pd.DataFrame(benchmark_rows)
print('VisualLocalization.net style localization scores')
print('All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)')
print('All conditions: ' + ' / '.join(benchmark_parts))
display(benchmark_df)

benchmark_json = results_dir / 'query_batch_visual_localization_thresholds.json'
benchmark_json.write_text(json.dumps(benchmark_rows, indent=2), encoding='utf-8')
print(f'Benchmark thresholds JSON: {benchmark_json}')

VisualLocalization.net style localization scores
All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)
All conditions: 18.9 / 40.5 / 100.0


,position_threshold_m,heading_threshold_deg,passed,total_queries,successful_evaluated_queries,percent_of_all_queries,percent_of_successful_queries
0,0.25,2.0,7,37,37,18.918919,18.918919
1,0.50,5.0,15,37,37,40.540541,40.540541
2,5.00,10.0,37,37,37,100.000000,100.000000


Benchmark thresholds JSON: ..\outputs\sim-19-may-bundle\query_batch_results_v2_19_may\query_batch_visual_localization_thresholds.json


## 7. Worst Successful Queries

In [8]:
if len(successful) > 0:
    worst = successful.sort_values('position_error_m', ascending=False).head(10)
    display(worst[['image_name', 'camera_name', 'capture_id', 'num_inliers', 'position_error_m', 'heading_error_deg']])
else:
    print('No successful queries to inspect.')

if len(failed) > 0:
    print('Failed queries:')
    display(failed[['image_name', 'camera_name', 'capture_id', 'error']])

,image_name,camera_name,capture_id,num_inliers,position_error_m,heading_error_deg
35,000036_front_f00001782.png,front,36,925,0.811012,3.731853
11,000012_front_f00000920.png,front,12,516,0.794412,0.045475
26,000027_front_f00001390.png,front,27,828,0.791780,0.012346
9,000010_front_f00000848.png,front,10,690,0.750979,0.023215
12,000013_front_f00000970.png,front,13,570,0.712147,0.190367
34,000035_front_f00001757.png,front,35,1350,0.710865,0.033377
8,000009_front_f00000823.png,front,9,335,0.706870,0.039157
25,000026_front_f00001363.png,front,26,685,0.696818,0.000070
20,000021_front_f00001215.png,front,21,381,0.687366,3.243842
7,000008_front_f00000794.png,front,8,590,0.676932,0.057878
